# 📒 Notebook 3 — Preprocessing: Kontinuitas Deret Waktu
> **Peran:** Senior Data Scientist | **Proyek:** SPK Pantau Pasar Sumbawa

---

## ⏱️ 1. Urgensi Deret Waktu yang Kontinu

Model prediksi berbasis deret waktu (seperti Random Forest Regressor) mengandalkan:
- **Lag features**: `Harga[t-1]`, `Harga[t-7]` — membutuhkan data setiap hari.
- **Rolling statistics**: rata-rata/standar deviasi 7–14 hari ke belakang.

Data harga mentah hanya mencatat **hari kerja pasar** — akhir pekan dan hari libur tidak ada. Jika tidak diisi, lag/rolling akan menghitung gap yang salah.

**Solusi:** Bangun **Master Kalender** harian penuh dan lakukan *cross join* dengan semua komoditas, lalu lakukan *left join* dengan data harga aktual. Baris tanpa harga akan diisi pada langkah berikutnya (forward fill di Notebook 4).

```
Master Kalender (2021-01-01 … max_date)   ×   Komoditas Unik
   ↓  Cross Join  →  grid [Tanggal, Komoditas]
   ↓  Left Join dengan dataset gabungan
   →  preprocessed_dataset.csv
      (setiap baris = setiap hari × setiap komoditas; harga bisa NaN)
```


In [1]:
import logging
from pathlib import Path
import pandas as pd

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("Preprocessing")

# ── Paths ─────────────────────────────────────────────────────────────────────
MERGED_DIR:    Path = Path("../processed_data/merged")
PROCESSED_DIR: Path = Path("../processed_data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Load Merged Dataset ───────────────────────────────────────────────────────
logger.info("Memuat dataset gabungan …")
merged: pd.DataFrame = pd.read_csv(
    MERGED_DIR / "dataset_all_merged.csv", parse_dates=["Tanggal"]
)
logger.info("Dataset dimuat: %s baris, %s kolom", *merged.shape)


10:44:12 [INFO] Memuat dataset gabungan …
10:44:13 [INFO] Dataset dimuat: 39601 baris, 4 kolom


## 📅 2. Membangun Master Kalender & Cross Join


In [2]:
# ── Master Kalender ───────────────────────────────────────────────────────────
START_DATE: str = "2021-01-01"
max_date: pd.Timestamp = merged["Tanggal"].max()

date_range = pd.date_range(start=START_DATE, end=max_date, freq="D")
commodities = merged["Komoditas"].unique()

logger.info("Rentang kalender : %s → %s  (%d hari)", START_DATE, max_date.date(), len(date_range))
logger.info("Komoditas unik   : %d", len(commodities))
logger.info("Total grid baris : %d × %d = %d", len(date_range), len(commodities),
            len(date_range) * len(commodities))

# ── Cross Join (Tanggal × Komoditas) ─────────────────────────────────────────
idx = pd.MultiIndex.from_product(
    [date_range, commodities], names=["Tanggal", "Komoditas"]
)
grid: pd.DataFrame = pd.DataFrame(index=idx).reset_index()

# ── Left Join dengan dataset gabungan ────────────────────────────────────────
preprocessed: pd.DataFrame = pd.merge(
    grid, merged, on=["Tanggal", "Komoditas"], how="left"
)

logger.info("Shape preprocessed (expected %d): %s",
            len(date_range) * len(commodities), preprocessed.shape)


10:44:13 [INFO] Rentang kalender : 2021-01-01 → 2026-11-10  (2140 hari)
10:44:13 [INFO] Komoditas unik   : 57
10:44:13 [INFO] Total grid baris : 2140 × 57 = 121980
10:44:13 [INFO] Shape preprocessed (expected 121980): (121980, 4)


## 💾 3. Simpan Output


In [3]:
output_path: Path = PROCESSED_DIR / "preprocessed_dataset.csv"
preprocessed.to_csv(output_path, index=False)
logger.info("✅ Preprocessed dataset disimpan → %s", output_path.resolve())

# ── Preview ───────────────────────────────────────────────────────────────────
missing_pct = 100 * preprocessed["Harga"].isna().sum() / len(preprocessed)
print(f"Baris total   : {len(preprocessed):,}")
print(f"Harga NaN     : {preprocessed['Harga'].isna().sum():,}  ({missing_pct:.1f}%) ← akan diisi di NB4")
preprocessed.head(10)


10:44:13 [INFO] ✅ Preprocessed dataset disimpan → C:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\processed_data\processed\preprocessed_dataset.csv


Baris total   : 121,980
Harga NaN     : 82,379  (67.5%) ← akan diisi di NB4


,Tanggal,Komoditas,Harga,Sumber
0,2021-01-01,bawang_merah,NaN,NaN
1,2021-01-01,bawang_putih_honan,NaN,NaN
2,2021-01-01,beras_medium,NaN,NaN
3,2021-01-01,beras_premium,NaN,NaN
4,2021-01-01,cabai_merah_besar,NaN,NaN
5,2021-01-01,cabai_merah_keriting,NaN,NaN
6,2021-01-01,cabai_rawit_merah,NaN,NaN
7,2021-01-01,daging_ayam_ras,NaN,NaN
8,2021-01-01,daging_sapi_paha_belakang,NaN,NaN
9,2021-01-01,daging_sapi_paha_depan,NaN,NaN
